# T06: Strukturierte Ausgaben

Eine Antwort in Fliesstext genügt für einen Chat, aber ein Compliance-Dashboard oder eine Datenbank braucht Felder: Produkt, EPD-Status, GWP-Wert, Quellen. Dieses Notebook lässt den RAG-Agenten JSON nach einem Schema zurückgeben und validiert es anschliessend.

1. Fliesstext und Quellenangaben über den Prompt
2. Ein JSON-Schema, das die API erzwingt (Structured Outputs)
3. Validierung des geparsten Ergebnisses
4. Umgang mit Fragen, die die Dokumente nicht beantworten

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos). Zuerst T01 ausführen: Dort entsteht der Vector Store, den die späteren Notebooks wiederverwenden.

In [ ]:
import sys

from loguru import logger
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

QUERY = "Which products have a third-party verified EPD, and what is their GWP?"
answer = await ask(agent, QUERY)

Die Fakten mögen stimmen, aber sie stecken in Fliesstext, den ein anderes Programm nicht verwenden kann.

## 1. Quellenangaben über den Prompt

Der Agent übergibt jeden Chunk als `<source id='...'>...</source>`. Ein Prompt kann das Modell bitten, diese IDs pro Fakt anzugeben. Nichts erzwingt das Format, und die IDs sind nichtssagende Chunk-IDs; `ask()` ordnet sie wieder den Dateien zu.

Die Ergänzung zu `SYSTEM_PROMPT` übersetzt:

> - Gib nach jedem Fakt die id seines <source>-Tags an, z. B. (source: 3f2a...).

In [ ]:
CITING_PROMPT = (
    SYSTEM_PROMPT
    + "\n- After every fact, cite the id of its <source> tag, e.g. (source: 3f2a...)."
)
citing_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=CITING_PROMPT,
)
answer = await ask(citing_agent, QUERY)

## 2. Structured Outputs

Mit den Structured Outputs von OpenAI beschränkt die API die Generierung auf ein JSON-Schema: Die Antwort lässt sich immer parsen und enthält alle Pflichtfelder. Wir definieren das Schema als Pydantic-Modell und übergeben sein JSON-Schema als `response_format`. Der `strict`-Modus verlangt, dass jedes Feld als Pflichtfeld aufgeführt ist und keine zusätzlichen Felder erlaubt sind (`extra="forbid"`); optionale Werte werden als `float | None` ausgedrückt.

In [ ]:
class Product(BaseModel):
    model_config = ConfigDict(extra="forbid")

    product_name: str
    has_verified_epd: bool
    gwp_kg_co2e: float | None = Field(
        description="GWP in kg CO2e per declared unit; null if not stated"
    )
    source_ids: list[str] = Field(description="ids of the <source> tags used")


class ProductCompliance(BaseModel):
    model_config = ConfigDict(extra="forbid")

    products: list[Product]


structured_llm = build_llm(
    "openai",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "ProductCompliance",
            "strict": True,
            "schema": ProductCompliance.model_json_schema(),
        },
    },
)
structured_agent = build_agent(
    vector_store,
    embedding_model,
    structured_llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

answer = await ask(structured_agent, QUERY)

## 3. Validierung

Das Schema garantiert die Struktur, nicht die Wahrheit. Das JSON in das Pydantic-Modell parsen und dann prüfen, was das Schema nicht prüfen kann: plausible Werte und Quell-IDs, die tatsächlich gefunden wurden (eine zitierte ID, die nie gefunden wurde, ist erfunden).

In [ ]:
def check(answer) -> ProductCompliance | None:
    try:
        result = ProductCompliance.model_validate_json(answer.content[0].text or "")
    except ValidationError as error:
        print(f"Invalid response: {error}")
        return None

    retrieved_ids = {source.id for source in answer.sources}
    for product in result.products:
        problems = []
        if product.gwp_kg_co2e is not None and product.gwp_kg_co2e <= 0:
            problems.append("non-positive GWP")
        if product.has_verified_epd and not product.source_ids:
            problems.append("verified EPD claimed without a source")
        if unknown := set(product.source_ids) - retrieved_ids:
            problems.append(f"{len(unknown)} cited id(s) were not retrieved")
        print(f"{product.product_name:<45} {'; '.join(problems) or 'ok'}")
    return result


result = check(answer)

## 4. Fragen ohne Antwort

Wenn die Dokumente die Antwort nicht enthalten, erzwingt das Schema trotzdem ein JSON-Objekt. Die richtige Ausgabe ist eine leere `products`-Liste, die der aufrufende Code explizit behandeln muss. Das Modell liefert unter Umständen stattdessen ein ähnliches Produkt (eine andere Palette); das erkennt das Schema nicht, deshalb prüfen, ob die zurückgegebenen Namen zur Frage passen.

In [ ]:
answer = await ask(
    structured_agent, "What is the GWP of the Lara pallet, and does it have an EPD?"
)
result = check(answer)
if result is not None:
    matches = [p for p in result.products if "lara" in p.product_name.lower()]
    others = [p.product_name for p in result.products if p not in matches]
    print(f"Lara pallet entries: {len(matches)}; other products returned: {others}")

## Zusammenfassung

| Ansatz | Garantiert | Einsetzen wenn |
|---|---|---|
| Anweisungen im Prompt | Nichts | Menschen lesen die Antwort |
| Structured Outputs (`response_format`) | Gültiges JSON nach Schema | Programme verarbeiten die Antwort |
| Pydantic-Validierung und Prüfungen | Typen, plausible Werte, belegte Quellen | Vor dem Speichern oder Anzeigen |